# 03 · Recurrent Neural Network: bidirectional LSTM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LekshmiSR/proteins-d1-hands-on/blob/main/notebooks/03_bilstm.ipynb)

**Idea.** A recurrent network reads the protein **one residue at a time**, like reading a sentence word by word. It keeps a **hidden state**, a vector that summarises everything read so far, and updates it at each step:

```
h0 → [cell] → h1 → [cell] → h2 → [cell] → h3 → ...
       ↑              ↑              ↑
      res 1         res 2          res 3
```

The same cell (same weights) is used at every step. In principle the hidden state can carry information from residues far away, unlike the fixed window of the MLP or CNN.

* **LSTM** (Long Short-Term Memory) is a recurrent cell with **gates** that decide what to keep, forget and output. It remembers over longer distances than a plain RNN.
* **Bidirectional**: whether a residue is in a helix depends on residues *before and after* it. So we run one LSTM left→right and a second one right→left, and join (concatenate) their hidden states at each residue.

Expected time on a Colab T4: about 8–12 minutes (recurrent networks are slower, because the steps along the chain cannot run in parallel).

## 0. Setup

Run the next cell first. It works in two places:

* **Google Colab**: it downloads this repository (code + helpers) into the Colab machine.
* **Your own computer** (Jupyter / VS Code): it just moves to the repository folder.

> **Colab tip:** use a GPU. Menu **Runtime → Change runtime type → T4 GPU → Save**. Do this *before* running cells.

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/LekshmiSR/proteins-d1-hands-on.git"
REPO_DIR = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")

if "google.colab" in sys.modules:                       # running in Colab
    if os.path.basename(os.getcwd()) != REPO_DIR:
        if not os.path.exists(REPO_DIR):
            subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
        os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":      # running locally from notebooks/
    os.chdir("..")

sys.path.insert(0, os.getcwd())
print("Working folder:", os.getcwd())

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from protein_dl import data, train, plots

train.set_seed(42)
device = train.get_device()

## 1. Settings

In [ ]:
EMB_DIM = 64         # size of the learned vector for each amino acid
HIDDEN = 128         # size of the LSTM hidden state (per direction)
N_LAYERS = 2         # stacked LSTM layers
DROPOUT = 0.2

EPOCHS = 10
BATCH_SIZE = 32
LR = 1e-3
MAX_LEN = 512
TRAIN_SUBSET = None  # e.g. 2000 for a quick test
SAVE_TO_DRIVE = False

## 2. Load the data

The same loaders as in notebook 00 (see there for every detail). We grab one batch to push through the model step by step.

In [ ]:
train_dl, valid_dl, test_dl = data.make_dataloaders("data", batch_size=BATCH_SIZE, max_len=MAX_LEN,
                                                     train_subset=TRAIN_SUBSET)
tokens, labels, lengths = next(iter(train_dl))
print("tokens :", tokens.shape, "  <- [B, L]")
print("labels :", labels.shape)
print("lengths:", lengths.shape)

## 3. Build the model step by step

### Step 1: an LSTM cell, one residue at a time

To really see the recurrence, we first run a single `LSTMCell` by hand over the first 5 residues of one protein. The LSTM keeps **two** vectors: the hidden state `h` (its output) and the cell state `c` (its long-term memory).

In [ ]:
embedding = nn.Embedding(data.VOCAB_SIZE, EMB_DIM, padding_idx=data.PAD_IDX)
cell = nn.LSTMCell(input_size=EMB_DIM, hidden_size=HIDDEN)

x = embedding(tokens[:1])                    # first protein only: [1, L, E]
h = torch.zeros(1, HIDDEN)                   # hidden state starts at zero
c = torch.zeros(1, HIDDEN)                   # cell state starts at zero
print("input of one step:", tuple(x[:, 0].shape), "  [1, E]")
for t in range(5):
    h, c = cell(x[:, t], (h, c))             # same cell, new residue, previous state
    print(f"after residue {t}: h {tuple(h.shape)}, c {tuple(c.shape)}, first values of h: {h[0, :4].detach().numpy().round(3)}")

Inside the cell, four small layers (the **gates**) are computed from the current residue and the previous `h`:

* **forget gate** `f`: how much of the old memory `c` to keep (0 = forget, 1 = keep)
* **input gate** `i` and **candidate** `g`: what new information to write into memory
* **output gate** `o`: how much of the memory to show as the new `h`

```
c_new = f * c_old + i * g
h_new = o * tanh(c_new)
```

That is why the weight matrix holds 4 × HIDDEN rows:

In [ ]:
print("weight_ih (input -> gates):", tuple(cell.weight_ih.shape), f"  [4*HIDDEN, E] = [4*{HIDDEN}, {EMB_DIM}]")
print("weight_hh (h -> gates)    :", tuple(cell.weight_hh.shape), f"  [4*HIDDEN, HIDDEN]")

### Step 2: `nn.LSTM` runs the loop for us, over the whole batch

`nn.LSTM` does the same thing as the loop above, but over all residues and all proteins in the batch, and much faster (on a GPU it uses NVIDIA's cuDNN).

In [ ]:
x = embedding(tokens)                                        # [B, L, E]
lstm_fwd = nn.LSTM(EMB_DIM, HIDDEN, batch_first=True)
out, (h_n, c_n) = lstm_fwd(x)
print("input          :", tuple(x.shape), "  [B, L, E]")
print("output         :", tuple(out.shape), "  [B, L, HIDDEN]  hidden state at EVERY residue")
print("final h        :", tuple(h_n.shape), "  [layers, B, HIDDEN]  only the last step (not needed here)")

### Step 3: bidirectional and stacked

With `bidirectional=True` PyTorch runs a second LSTM from right to left and concatenates both outputs, so each residue gets `2 × HIDDEN` numbers: a summary of everything to its left **and** everything to its right.
With `num_layers=2`, a second LSTM reads the output of the first one.

In [ ]:
bilstm = nn.LSTM(EMB_DIM, HIDDEN, num_layers=N_LAYERS, batch_first=True, bidirectional=True, dropout=DROPOUT)
out, _ = bilstm(x)
print("BiLSTM output:", tuple(out.shape), f"  [B, L, 2*HIDDEN] = [B, L, {2 * HIDDEN}]")

### Step 4: the padding problem and packed sequences

The right→left LSTM of a short protein would start reading at the **end of the padding**, so its memory would be filled with padding before it reaches the real residues. That would make the prediction for a protein depend on which other proteins share its batch.

The fix: **pack** the batch. `pack_padded_sequence` tells the LSTM the true length of each protein, so it skips padding completely. Afterwards `pad_packed_sequence` turns the result back into a normal `[B, L, 2*HIDDEN]` tensor (with zeros at padded positions).

Let us prove that it matters: run one protein alone, then inside a padded batch, with and without packing.

In [ ]:
bilstm.eval()                                   # turn off dropout for an exact comparison
with torch.no_grad():
    i = int(lengths.argmin())                   # the shortest protein in the batch has the most padding
    L = int(lengths[i])
    alone, _ = bilstm(embedding(tokens[i:i+1, :L]))                       # no padding at all

    in_batch, _ = bilstm(x)                                                # padded, not packed
    packed = nn.utils.rnn.pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
    out_packed, _ = bilstm(packed)
    out_packed, _ = nn.utils.rnn.pad_packed_sequence(out_packed, batch_first=True, total_length=x.shape[1])

print(f"protein {i}: {L} residues, padded to {x.shape[1]}")
print("max difference vs. alone, padded batch :", (alone[0] - in_batch[i, :L]).abs().max().item())
print("max difference vs. alone, packed batch :", (alone[0] - out_packed[i, :L]).abs().max().item())

Without packing the outputs differ (the backward direction was disturbed by the padding). With packing they are identical up to rounding. Our model always packs.

### Step 5: classification head

A `Linear(2*HIDDEN, 3)` turns each residue's 256 numbers into 3 logits.

### The model class

In [ ]:
class BiLSTM(nn.Module):
    def __init__(self, emb_dim=EMB_DIM, hidden=HIDDEN, n_layers=N_LAYERS, dropout=DROPOUT):
        super().__init__()
        self.embedding = nn.Embedding(data.VOCAB_SIZE, emb_dim, padding_idx=data.PAD_IDX)
        self.lstm = nn.LSTM(emb_dim, hidden, num_layers=n_layers, batch_first=True,
                            bidirectional=True, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(2 * hidden, data.NUM_CLASSES)

    def forward(self, tokens, lengths):
        x = self.embedding(tokens)                                          # [B, L, E]
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=tokens.shape[1])
        return self.head(self.dropout(out))                                 # [B, L, 3]


model = BiLSTM().to(device)
print(model)
print(f"\nTrainable parameters: {train.count_parameters(model):,}")
with torch.no_grad():
    print("output shape on our batch:", tuple(model(tokens.to(device), lengths).shape))

## Train

`train.train_model` runs the loop from notebook 01: for each epoch it trains on all training batches, then measures loss and Q3 on the validation set, and remembers the weights of the best epoch. 

In [ ]:
train.set_seed(42)
history = train.train_model(model, train_dl, valid_dl, device, epochs=EPOCHS, lr=LR)

In [ ]:
plots.plot_history(history, "Bidirectional LSTM")

**Reading the curves:** training loss should go down steadily. If validation loss starts going *up* while training loss keeps going down, the model is **overfitting** (memorising the training proteins). The best-epoch weights protect us from that.

## Evaluate on the test set (CB513)

The model now has the weights from its best validation epoch. We run it **once** on CB513, the independent test set.

In [ ]:
val_metrics = train.evaluate(model, valid_dl, device)
test_metrics = train.evaluate(model, test_dl, device)

print(f"Validation Q3: {val_metrics['q3_accuracy']:.4f}")
print(f"CB513 Q3     : {test_metrics['q3_accuracy']:.4f}   macro F1: {test_metrics['macro_f1']:.4f}")
print()
print("Per-class scores on CB513:")
for cls, s in test_metrics["per_class"].items():
    print(f"  {cls:6s} precision {s['precision']:.3f}  recall {s['recall']:.3f}  F1 {s['f1']:.3f}  ({s['support']} residues)")

In [ ]:
plots.plot_confusion(test_metrics["confusion_matrix"], "BiLSTM on CB513 (rows sum to 1)")

**How to read the confusion matrix:** each row is a true class and shows where its residues ended up. The diagonal is the recall of each class. Off-diagonal cells show the typical mistakes (strands predicted as coil is usually the biggest one).

Let us look at a real prediction for one CB513 protein:

In [ ]:
trues, preds, _ = train.predict(model, test_dl, device)
i = 100                                           # try other numbers!
x = np.array(test_dl.dataset.tokens[i]); y = np.array(test_dl.dataset.labels[i])
seq = "".join(data.AMINO_ACIDS[t - 1] if 1 <= t <= 20 else "X" for t in x[y != data.IGNORE_INDEX])
print(f"protein {i}: {len(seq)} scored residues, accuracy {(trues[i] == preds[i]).mean():.2f}\n")
plots.show_prediction(seq, trues[i], preds[i])

## Save the results

The results (metrics, training curves, settings) are written to `results/bilstm.json`. Notebook 05 reads these files to compare all models.

In Colab, files disappear when the session ends. Set `SAVE_TO_DRIVE = True` in the settings cell at the top to also keep a copy in your Google Drive (Colab will ask for permission).

In [ ]:
drive_dir = train.get_drive_dir(SAVE_TO_DRIVE)
config = dict(EPOCHS=EPOCHS, BATCH_SIZE=BATCH_SIZE, LR=LR, MAX_LEN=MAX_LEN, TRAIN_SUBSET=TRAIN_SUBSET)
results = train.save_results("bilstm", model, history, val_metrics, test_metrics, config=config, drive_dir=drive_dir)

## What did we learn?

* The BiLSTM sees, in principle, the **whole** protein from both sides. On this task it is usually the strongest of the "from scratch" models, especially for strands, which depend on longer-range context.
* The price is speed: the residues must be processed one after another, so it is the slowest model to train.

The **Transformer** in the next notebook also sees the whole protein, but in parallel, by letting every residue directly "attend" to every other residue.

## Exercises

1. Set `bidirectional=False` (and change the head to `Linear(hidden, 3)`). How much does reading only left→right hurt?
2. Replace `nn.LSTM` by `nn.GRU` (a simpler recurrent cell; it returns `out, h` instead of `out, (h, c)`). Compare speed and Q3.
3. Try `N_LAYERS = 1` and `N_LAYERS = 3`.